In [2]:
import sys
import pyspark.sql.functions as F
from src.config import data_paths, filter_map
from src.data_ingestion import DataLoader,DataMerger,Preprocessor
from src.logger import logger
from src.exception import M5Exception

In [3]:
#DataLoader initialization
try:
    data_loader_obj = DataLoader(data_paths)
    logger.info("DataLoader initialized successfully!")
except Exception as e:
    logger.info("DataLoader failed!")
    raise M5Exception(str(e),sys) from None


In [4]:
#get input raw datasets
try:
    input_data = data_loader_obj.get_input_data()
    logger.info("data_loader_obj.get_input_data ran successfully!")
except Exception as e:
    logger.info("data_loader_obj.get_input_data run failed!")
    raise M5Exception(str(e),sys) from None

In [5]:
input_data['sales_wide'].count()

30490

In [6]:
#Data Transformer initialization
try:
    data_merger_obj = DataMerger(input_data)
    logger.info("DataMerger obj initialized successfully!")
except Exception as e:
    logger.info("DataMerger failed!")
    raise M5Exception(str(e),sys) from None

In [7]:
#merge raw datasets
try:
    sales_long, calendar,price,joined_data = data_merger_obj.merge_raw_datasets()
    logger.info("data_merger_obj.merge_raw_datasets ran successfully!")
except Exception as e:
    logger.info("data_merger_obj.merge_raw_datasets run failed!")
    raise M5Exception(str(e),sys) from None

In [8]:
try:
    preprocessor_obj = Preprocessor(joined_data,filter_map)
    logger.info("Preprocessor obj initialized successfully!")
except Exception as e:
    logger.info("Preprocessor failed!")
    raise M5Exception(str(e),sys) from None

In [11]:
joined_data.show()

+--------+--------+-------------+----+------------------+---------+-------+--------+-----+----------+---------+----+-----+----+-------------+------------+------------+------------+-------+-------+-------+----------+
|wm_yr_wk|store_id|      item_id|   d|                id|  dept_id| cat_id|state_id|sales|      date|  weekday|wday|month|year| event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|sell_price|
+--------+--------+-------------+----+------------------+---------+-------+--------+-----+----------+---------+----+-----+----+-------------+------------+------------+------------+-------+-------+-------+----------+
|   11101|    TX_2|HOBBIES_2_105| d_1|HOBBIES_2_105_TX_2|HOBBIES_2|HOBBIES|      TX|    0|2011-01-29| Saturday|   1|    1|2011|         NULL|        NULL|        NULL|        NULL|      0|      0|      0|      NULL|
|   11101|    TX_2|HOBBIES_2_105| d_2|HOBBIES_2_105_TX_2|HOBBIES_2|HOBBIES|      TX|    0|2011-01-30|   Sunday|   2|    1|2011|         

In [10]:

try:
    weekly_sales_data = preprocessor_obj.process()
    logger.info("preprocessor_obj.process() successful!")
except Exception as e:
    logger.info("preprocessor_obj.process() failed!")
    raise M5Exception(str(e),sys) from None

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\m5_forecasting\m5-demand-forecasting\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\m5_forecasting\m5-demand-forecasting\.venv\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "c:\m5_forecasting\m5-demand-forecasting\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    resp

KeyboardInterrupt: 

In [8]:
weekly_sales_data.show(5)

+----------------+-----------+-------+------+--------+--------+--------+------------+----------+------------+------------+------------+-----------------+-------------------+-----------------+--------------------+----------------+----------------------+------------------+-------------------------+----------------------+--------------------+---------------------+--------------+-----------------+-------------------+--------------------+-----------------+----------------------+-----------------+-------------------+-------------------+---------------------+-----------------+---------------------+-------------------+---------------------------+-----------------+---------------------+---------------+-----------------------+-------------------+
|              id|    item_id|dept_id|cat_id|store_id|state_id|wm_yr_wk|weekly_sales|sell_price|snap_CA_days|snap_TX_days|snap_WI_days|f_event_halloween|f_event_veteransday|f_event_christmas|f_event_thanksgiving|f_event_laborday|f_event_orthodoxeaster|f